In [5]:
import sys
import pandas as pd
import numpy as np

In [6]:
# предварительный сбор данных для проверки
file_old = 'music_old.csv'
file_new = 'music_new.csv'

df_old = pd.read_csv(file_old)
df_new = pd.read_csv(file_new)

old_loudness = df_old['loudness'].dropna()
new_loudness = df_new['loudness'].dropna()

In [ ]:
# считаем PSI
def calculate_psi(data_old, data_new, num_bins=20):
    # построение бинов на основе старых данных
    bin_edges = []
    for i in range(num_bins + 1):
        percentile_val = (100 / num_bins) * i
        bin_edges.append(np.percentile(data_old, percentile_val))

    bin_edges[-1] = bin_edges[-1] + 0.0001
    
    # количество значений в каждом бине для старых данных
    counts_old = [0] * num_bins
    for val in data_old:
        for i in range(num_bins):
            if bin_edges[i] <= val < bin_edges[i + 1]:
                counts_old[i] = counts_old[i] + 1
                break
    
    # количество значений в каждом бине для новых данных
    counts_new = [0] * num_bins
    for val in data_new:
        for i in range(num_bins):
            if bin_edges[i] <= val < bin_edges[i + 1]:
                counts_new[i] = counts_new[i] + 1
                break
    # доля треков в бине для старых данных
    old_percents = []
    for count in counts_old:
        old_percents.append(count / len(data_old))
    
    # доля треков в бине для новых данных
    new_percents = []
    for count in counts_new:
        new_percents.append(count / len(data_new))
    
    # замена нулевых значений на маленькое число для избежания деления на ноль
    for i in range(num_bins):
        if old_percents[i] == 0:
            old_percents[i] = 0.0001
        if new_percents[i] == 0:
            new_percents[i] = 0.0001
            
    # считаем суммарный PSI по всем записям
    psi_val = 0
    for i in range(num_bins):
        psi_val = psi_val + (new_percents[i] - old_percents[i]) * np.log(new_percents[i] / old_percents[i])
    
    return psi_val

In [8]:
# выводы по PSI
psi_value = calculate_psi(old_loudness, new_loudness, 20)
print(f"{psi_value=}")
if psi_value < 0.1:
    print("PSI < 0.1: дрейфа данных нет")
elif psi_value < 0.2:
    print("0.1 <= PSI < 0.2: есть дрейф данных")
else:
    print("PSI >= 0.2: сильный дрейф данных")

psi_value=0.23527298577359163
PSI >= 0.2: сильный дрейф данных
